In [1]:
import pandas as pd
import numpy as np

In [2]:
lora_metrics = {
    "pipeline": "LoRA_Llama3_FEINA30",
    "model_family": "llama3",
    "method_type": "lora",
    "split_eval": "test",
    "sari": 38.085834,
    "fernandez_huerta_pred": 72.750557,
    "fernandez_huerta_src": 76.029628,
    "fernandez_huerta_delta": -3.279071,
    "compression_ratio_eval": 0.876656,
    "exact_copy": 0.210084,
}

In [3]:
prompt_metrics = {
    "pipeline": "PROMPT_Llama3_cfg3_R1",
    "model_family": "llama3",
    "method_type": "prompt_based",
    "split_eval": "test",

    "sari": 30.320823,
    "fernandez_huerta_pred": 77.364982,
    "fernandez_huerta_src": 76.029628,
    "fernandez_huerta_delta": 1.335355,
    "compression_ratio_eval": 0.819666,
    "exact_copy": 0.0,
}

In [4]:
comparison_df = pd.DataFrame([prompt_metrics, lora_metrics])
display(comparison_df)

,pipeline,model_family,method_type,split_eval,sari,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,exact_copy
0,PROMPT_Llama3_cfg3_R1,llama3,prompt_based,test,30.320823,77.364982,76.029628,1.335355,0.819666,0.000000
1,LoRA_Llama3_FEINA30,llama3,lora,test,38.085834,72.750557,76.029628,-3.279071,0.876656,0.210084


In [5]:
main_metric_cols = [
    "pipeline",
    "model_family",
    "method_type",
    "split_eval",
    "sari",
    "fernandez_huerta_pred",
    "fernandez_huerta_src",
    "fernandez_huerta_delta",
    "compression_ratio_eval",
    "exact_copy",
]

comparison_main_df = comparison_df[main_metric_cols].copy()
display(comparison_main_df)

,pipeline,model_family,method_type,split_eval,sari,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,exact_copy
0,PROMPT_Llama3_cfg3_R1,llama3,prompt_based,test,30.320823,77.364982,76.029628,1.335355,0.819666,0.000000
1,LoRA_Llama3_FEINA30,llama3,lora,test,38.085834,72.750557,76.029628,-3.279071,0.876656,0.210084


In [6]:
prompt_row = comparison_df[comparison_df["method_type"] == "prompt_based"].iloc[0]
lora_row = comparison_df[comparison_df["method_type"] == "lora"].iloc[0]

print("Prompt-based:")
display(prompt_row.to_frame().T)

print("LoRA:")
display(lora_row.to_frame().T)

Prompt-based:


,pipeline,model_family,method_type,split_eval,sari,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,exact_copy
0,PROMPT_Llama3_cfg3_R1,llama3,prompt_based,test,30.320823,77.364982,76.029628,1.335355,0.819666,0.0


LoRA:


,pipeline,model_family,method_type,split_eval,sari,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,exact_copy
1,LoRA_Llama3_FEINA30,llama3,lora,test,38.085834,72.750557,76.029628,-3.279071,0.876656,0.210084


In [7]:
delta_metrics = {
    "delta_sari": lora_row["sari"] - prompt_row["sari"],
    "delta_fernandez_huerta_pred": lora_row["fernandez_huerta_pred"] - prompt_row["fernandez_huerta_pred"],
    "delta_fernandez_huerta_delta": lora_row["fernandez_huerta_delta"] - prompt_row["fernandez_huerta_delta"],
    "delta_compression_ratio_eval": lora_row["compression_ratio_eval"] - prompt_row["compression_ratio_eval"],
    "delta_exact_copy": lora_row["exact_copy"] - prompt_row["exact_copy"],
}

delta_df = pd.DataFrame([delta_metrics])
display(delta_df)

,delta_sari,delta_fernandez_huerta_pred,delta_fernandez_huerta_delta,delta_compression_ratio_eval,delta_exact_copy
0,7.765011,-4.614425,-4.614426,0.05699,0.210084


In [8]:
comparison_df["leader_score_simple"] = comparison_df["sari"]
display(comparison_df[["pipeline", "method_type", "leader_score_simple"]])

,pipeline,method_type,leader_score_simple
0,PROMPT_Llama3_cfg3_R1,prompt_based,30.320823
1,LoRA_Llama3_FEINA30,lora,38.085834


In [9]:
def compare_metric(lora_value, prompt_value, higher_is_better=True, tol=1e-9):
    if pd.isna(lora_value) or pd.isna(prompt_value):
        return "incompleto"

    diff = lora_value - prompt_value

    if abs(diff) <= tol:
        return "empate"

    if higher_is_better:
        return "gana_lora" if diff > 0 else "gana_prompt"
    else:
        return "gana_lora" if diff < 0 else "gana_prompt"

In [10]:
comparison_interpretation = {
    "sari_result": compare_metric(
        lora_row["sari"],
        prompt_row["sari"],
        higher_is_better=True
    ),
    "fernandez_huerta_pred_result": compare_metric(
        lora_row["fernandez_huerta_pred"],
        prompt_row["fernandez_huerta_pred"],
        higher_is_better=True
    ),
    "exact_copy_result": compare_metric(
        lora_row["exact_copy"],
        prompt_row["exact_copy"],
        higher_is_better=False
    ),
}

comparison_interpretation_df = pd.DataFrame([comparison_interpretation])
display(comparison_interpretation_df)

,sari_result,fernandez_huerta_pred_result,exact_copy_result
0,gana_lora,gana_prompt,gana_prompt


In [11]:
final_report_df = pd.DataFrame([
    {
        "pipeline": prompt_row["pipeline"],
        "method_type": prompt_row["method_type"],
        "sari": prompt_row["sari"],
        "fh_pred": prompt_row["fernandez_huerta_pred"],
        "fh_delta": prompt_row["fernandez_huerta_delta"],
        "compression_ratio": prompt_row["compression_ratio_eval"],
        "exact_copy": prompt_row["exact_copy"],
    },
    {
        "pipeline": lora_row["pipeline"],
        "method_type": lora_row["method_type"],
        "sari": lora_row["sari"],
        "fh_pred": lora_row["fernandez_huerta_pred"],
        "fh_delta": lora_row["fernandez_huerta_delta"],
        "compression_ratio": lora_row["compression_ratio_eval"],
        "exact_copy": lora_row["exact_copy"],
    },
])

display(final_report_df)

,pipeline,method_type,sari,fh_pred,fh_delta,compression_ratio,exact_copy
0,PROMPT_Llama3_cfg3_R1,prompt_based,30.320823,77.364982,1.335355,0.819666,0.000000
1,LoRA_Llama3_FEINA30,lora,38.085834,72.750557,-3.279071,0.876656,0.210084


In [13]:
print(f"""
Resumen de comparación final:

Se comparó el mejor pipeline prompt-based contra el modelo ajustado con LoRA
sobre el mismo conjunto de prueba del 30 por ciento.

Resultados del prompt-based:
- SARI: {prompt_row['sari']:.6f}
- Fernández-Huerta pred: {prompt_row['fernandez_huerta_pred']:.6f}
- Fernández-Huerta delta: {prompt_row['fernandez_huerta_delta']:.6f}
- Compression ratio: {prompt_row['compression_ratio_eval']:.6f}
- Exact copy: {prompt_row['exact_copy']:.6f}

Resultados del LoRA:
- SARI: {lora_row['sari']:.6f}
- Fernández-Huerta pred: {lora_row['fernandez_huerta_pred']:.6f}
- Fernández-Huerta delta: {lora_row['fernandez_huerta_delta']:.6f}
- Compression ratio: {lora_row['compression_ratio_eval']:.6f}
- Exact copy: {lora_row['exact_copy']:.6f}
""")


Resumen de comparación final:

Se comparó el mejor pipeline prompt-based contra el modelo ajustado con LoRA
sobre el mismo conjunto de prueba del 30 por ciento.

Resultados del prompt-based:
- SARI: 30.320823
- Fernández-Huerta pred: 77.364982
- Fernández-Huerta delta: 1.335355
- Compression ratio: 0.819666
- Exact copy: 0.000000

Resultados del LoRA:
- SARI: 38.085834
- Fernández-Huerta pred: 72.750557
- Fernández-Huerta delta: -3.279071
- Compression ratio: 0.876656
- Exact copy: 0.210084

